# Header

In [ ]:
import importlib
import xarray_sf_funcs as xsfuncs
from dask.distributed import Client
import dask
import os
import re
import glob
import warnings
import logging

# Reload the xsfuncs module
importlib.reload(xsfuncs)

warnings.filterwarnings("ignore")


In [ ]:
region_dict = {
    'acc': {'lat_north': -53, 'lat_south': -57.5, 'lon_east': 158, 'lon_west': 148, 'reduce_xd_num': 1},
    'nwpacific': {'lat_north': 23, 'lat_south': 19, 'lon_east': 137, 'lon_west': 132, 'reduce_xd_num': 1},
    'capebasin': {'lat_north': -41.01421, 'lat_south': -44.99279, 'lon_east': 16, 'lon_west': 11, 'reduce_xd_num': 1},
    'newcaledonia': {'lat_north': -22, 'lat_south': -26, 'lon_east': 171, 'lon_west': 166, 'reduce_xd_num': 1},
    'nwaustralia': {'lat_north': -11, 'lat_south': -15, 'lon_east': 125, 'lon_west': 120, 'reduce_xd_num': 1},
    'westatlantic': {'lat_north': 38.7, 'lat_south': 32.7, 'lon_east': -73, 'lon_west': -76, 'reduce_xd_num': 1},
    'labradorsea': {'lat_north': 63.79824, 'lat_south': 59.5549, 'lon_east': -58.52856, 'lon_west': -63.61784, 'reduce_xd_num': 1},
    # 'Florida': {'lat_north': 29, 'lat_south': 24, 'lon_east': -77, 'lon_west': -82, 'reduce_xd_num': 1},
    # 'GrandBanks': {'lat_north': 40, 'lat_south': 35, 'lon_east': -45, 'lon_west': -50, 'reduce_xd_num': 1},
    # 'Arbic': {'lat_north': 43, 'lat_south': 27.5, 'lon_east': -40, 'lon_west': -60, 'reduce_xd_num': 1},
    # 'Equator': {'lat_north': 5, 'lat_south': -5, 'lon_east': -10, 'lon_west': -35, 'reduce_xd_num': 1},
    # 'Interior': {'lat_north': 32, 'lat_south': 22, 'lon_east': -30, 'lon_west': -42, 'reduce_xd_num': 1},
    # 'GulfStream': {'southwest': (-80, 25), 'southeast': (-35, 25), 'northwest': (-80, 39), 'northeast': (-35, 49), 'reduce_xd_num': 1}
}

# SWOT L3 8 Regions

*THE ONLY COMPLETE SCIENCE PHASE CYCLES ARE 001-027*

In [3]:
import gc
import os
import re
import glob
import time
import signal
import warnings
import logging

# Configure Dask memory settings
dask.config.set({
    "distributed.worker.memory.target": 0.7,
    "distributed.worker.memory.spill": 0.8,
    "distributed.worker.memory.pause": 0.85,
    "distributed.worker.memory.terminate": 0.95,
    "logging.distributed": "error",
})

def create_client():
    """Helper function to create a new Dask client."""
    return Client(
        n_workers=8,
        threads_per_worker=1,
        memory_limit="48GB",
        silence_logs=logging.WARNING,
    )

def reset_client(client):
    """Safely reset Dask client without crashing on teardown timeouts."""
    try:
        client.restart(wait_for_workers=False)
        print("Dask client restarted successfully")
        return client
    except Exception as restart_err:
        print(f"Client restart failed ({restart_err}); creating a fresh client...")

        try:
            client.shutdown()
        except Exception as shutdown_err:
            print(f"Client shutdown warning (safe to ignore): {shutdown_err}")

        try:
            client.close()
        except Exception as close_err:
            print(f"Client close warning (safe to ignore): {close_err}")

        gc.collect()
        new_client = create_client()
        print("Created a fresh Dask client")
        return new_client

class RunTimeoutError(TimeoutError):
    pass

def _timeout_handler(signum, frame):
    raise RunTimeoutError("Run exceeded timeout window without completion")

# macOS supports SIGALRM
signal.signal(signal.SIGALRM, _timeout_handler)

# ---------- Tunables ----------
STALL_TIMEOUT_MIN = 20          # hard timeout per region/cycle attempt
STALL_TIMEOUT_SEC = STALL_TIMEOUT_MIN * 60
MAX_RETRIES = 2                 # retries per region/cycle before skip
run_new = False                 # False => skip existing completed outputs
# -----------------------------

USE_TRY_EXCEPT = True
client = create_client()

region_skips = [
    # "acc", "already processed",
    # "nwpacific", "already processed",
    # "capebasin", "already processed",
    # "nwaustralia", "already processed",
    # "westatlantic", "already processed",
]

problematic_region_cycles = []
problematic_log_path = "problematic_region_cycles_swot.log"

parent_dir = "/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/forward"
output_dir = "data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-04-06"
all_dirs = [d for d in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, d))]

cycle_nums = sorted(
    re.search(r"cycle_(\d+)", d).group(1).zfill(3)
    for d in all_dirs if re.search(r"cycle_(\d+)", d)
)
cycle_nums = [num for num in cycle_nums if 1 <= int(num) <= 100]

# Optional start point:
# cycle_nums = [num for num in cycle_nums if int(num) >= 10]

print(f"Cycle numbers: {cycle_nums}")
print(f"run_new={run_new} (False means completed outputs will be skipped)")

for idx, cycnum in enumerate(cycle_nums):
    print(f"\nProcessing cycle: {cycnum}")

    # periodic proactive reset
    if idx > 0 and idx % 10 == 0:
        print(f"Restarting Dask client after {idx} iterations...")
        client = reset_client(client)

    print(f"Dask dashboard link: {client.dashboard_link}")

    for region, params in region_dict.items():
        if region in region_skips:
            print(f"Skipping region: {region}, {region_skips[region_skips.index(region) + 1]}")
            continue

        reduce_xd_num = params["reduce_xd_num"]
        lat_north = params["lat_north"]
        lat_south = params["lat_south"]
        lon_east = params["lon_east"]
        lon_west = params["lon_west"]

        # convert negative longitudes to 0..360
        if lon_west < 0:
            lon_west += 360
        if lon_east < 0:
            lon_east += 360

        print(f"Processing region: {region} with reduce_xd_num: {reduce_xd_num}")

        region_output_dir = os.path.join(output_dir, region)
        os.makedirs(region_output_dir, exist_ok=True)

        def matching_outputs():
            return [
                f for f in os.listdir(region_output_dir)
                if (
                    f.endswith(".nc")
                    and "timemean_removed_Bessels_tapered_CG" in f
                    and f"SWOT_L3_{region}_cycle_{cycnum}" in f
                )
            ]

        # skip completed
        if not run_new:
            existing_files = matching_outputs()
            if existing_files:
                print(f"Skipping cycle {cycnum} for region {region} as output file already exists: {existing_files[0]}")
                continue

        def run_once():
            print(f"Processing cycle {cycnum} for region {region}")
            before_files = set(matching_outputs())

            signal.alarm(STALL_TIMEOUT_SEC)
            try:
                with warnings.catch_warnings(record=True) as caught_warnings:
                    warnings.simplefilter("always")
                    xsfuncs.compute_and_save_SWOT_L3(
                        region_name=region,
                        cycle_num=cycnum,
                        lat_north=lat_north,
                        lat_south=lat_south,
                        lon_east=lon_east,
                        lon_west=lon_west,
                        CG=True,
                        reduce_xd_num=reduce_xd_num,
                        scalar=None,
                        Bessels=True,
                        LLL=False,
                        remove_timemean=True,
                        taper_SF=True,
                    )

                    # Optional: escalate memory warning strings if emitted as warnings
                    for w in caught_warnings:
                        if "Unmanaged memory use is high" in str(w.message):
                            raise RuntimeError("Unmanaged memory warning")
            finally:
                signal.alarm(0)

            client.run(gc.collect)

            after_files = set(matching_outputs())
            if len(after_files - before_files) == 0 and len(after_files) == 0:
                raise RuntimeError("No matching output file detected after computation")

        retries = 0
        while retries < MAX_RETRIES:
            try:
                print(f"Attempt {retries + 1}/{MAX_RETRIES} for cycle {cycnum}, region {region}")
                run_once()
                break

            except RunTimeoutError as e:
                retries += 1
                print(f"Timeout for region {region}, cycle {cycnum}: {e}")
                client = reset_client(client)
                gc.collect()

                if retries >= MAX_RETRIES:
                    reason = f"timeout after {MAX_RETRIES} retries"
                    print(f"Skipping cycle {cycnum} for region {region}: {reason}")
                    problematic_region_cycles.append((cycnum, region, reason))
                    with open(problematic_log_path, "a") as log_file:
                        log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")

            except RuntimeError as e:
                retries += 1
                print(f"RuntimeError for region {region}, cycle {cycnum}: {e}")
                client = reset_client(client)
                gc.collect()

                if retries >= MAX_RETRIES:
                    reason = f"runtime: {e}"
                    print(f"Skipping cycle {cycnum} for region {region}: {reason}")
                    problematic_region_cycles.append((cycnum, region, reason))
                    with open(problematic_log_path, "a") as log_file:
                        log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")
                else:
                    print(f"Retrying cycle {cycnum} for region {region}...")

            except Exception as e:
                retries += 1
                print(f"Exception for region {region}, cycle {cycnum}: {e}")
                client = reset_client(client)
                gc.collect()

                if retries >= MAX_RETRIES:
                    reason = f"exception: {e}"
                    print(f"Skipping cycle {cycnum} for region {region}: {reason}")
                    problematic_region_cycles.append((cycnum, region, reason))
                    with open(problematic_log_path, "a") as log_file:
                        log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")
                else:
                    print(f"Retrying cycle {cycnum} for region {region}...")

print("\nRun complete.")
if problematic_region_cycles:
    print("Problematic cycle/region pairs:")
    for cyc, reg, reason in problematic_region_cycles:
        print(f"  - cycle {cyc}, region {reg}: {reason}")
    print(f"Logged problematic pairs to {problematic_log_path}")
else:
    print("No problematic cycle/region pairs recorded.")

# Final cleanup
try:
    client.shutdown()
except Exception as e:
    print(f"Final client shutdown warning (safe to ignore): {e}")

try:
    client.close()
except Exception as e:
    print(f"Final client close warning (safe to ignore): {e}")

Cycle numbers: ['032', '033', '034', '035', '036', '037', '038', '039', '040', '041', '042', '043', '044', '045', '046', '047', '048']
run_new=False (False means completed outputs will be skipped)

Processing cycle: 032
Dask dashboard link: http://127.0.0.1:61695/status
Processing region: acc with reduce_xd_num: 1
Skipping cycle 032 for region acc as output file already exists: 20260407_034524_SWOT_L3_acc_cycle_032_timemean_removed_Bessels_tapered_CG.nc
Processing region: nwpacific with reduce_xd_num: 1
Skipping cycle 032 for region nwpacific as output file already exists: 20260407_034546_SWOT_L3_nwpacific_cycle_032_timemean_removed_Bessels_tapered_CG.nc
Processing region: capebasin with reduce_xd_num: 1
Skipping cycle 032 for region capebasin as output file already exists: 20260407_034615_SWOT_L3_capebasin_cycle_032_timemean_removed_Bessels_tapered_CG.nc
Processing region: newcaledonia with reduce_xd_num: 1
Skipping cycle 032 for region newcaledonia as output file already exists: 2026

2026-04-07 18:04:07,995 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:61716' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'compute_asf-8cbd4757-20e0-462f-8cce-1968e2e556f0', 'compute_asf-3040d620-3ca6-4261-b4ea-6accf6ce5955', 'compute_asf-b6af0284-fcaa-49eb-8647-7af41dec396d', 'compute_asf-b181bb44-f3f9-4265-bf0e-bd8bb28b7e90', 'compute_asf-ade4d04b-a5bb-46c3-b205-83a855ee3e93', 'compute_asf-c9e1f765-3c3a-4244-8b56-93c32e0ce56a', 'compute_asf-793b20a0-eb23-4e3d-bc9e-4a492ff9ca92', 'compute_asf-e3da3109-bbbb-425b-9c14-6df7c92852dd', 'compute_asf-63bb4f27-1bad-4f0e-b043-36dc42853059', 'compute_asf-469ad2af-d6d5-43a6-8211-2b227d71cb86', 'compute_asf-a6d49231-12ac-4844-ae14-cb403a47dc20', 'compute_asf-fbf60f6c-da7e-47ed-bec8-cb7031bf45aa', 'compute_asf-a073c2ac-6e87-4098-818d-166ee009e4f9', 'compute_asf-5744400b-2bf9-4f0f-8fca-d9d2213fa493', 'compute_asf-adbeb4df-c52c-48c8-b016-1d9e901b19e2', 'compute_asf-3eb7f0d6-168f-4f6

Computed ASFs
Formatted ASFs without scalar
Computed Bessel functions
Delayed coarse graining
Set up coarse graining
Computed coarse graining
Coarse graining formatted
Saved to data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-04-06/labradorsea/20260407_180454_SWOT_L3_labradorsea_cycle_034_timemean_removed_Bessels_tapered_CG.nc

Processing cycle: 035
Dask dashboard link: http://127.0.0.1:61695/status
Processing region: acc with reduce_xd_num: 1
Attempt 1/2 for cycle 035, region acc
Processing cycle 035 for region acc
Loading data from /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/forward/cycle_035/*.nc
Defined file paths with glob
Computing the maximum number of lines across all datasets
Number of datasets: 579
Maximum number of lines: 9860
Loading datasets and padding them to the maximum number of lines
Starting to compute padded datasets in parallel
Padded datasets
Filtered datasets by region
Removed time mean from dataset
Calculated advection
Computed ASFs
Formatted

2026-04-07 18:56:39,747 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:61715' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'compute_asf-35285434-b15b-417f-83b8-bab378e1b5e0', 'compute_asf-25466a91-8811-4909-a60b-341c1d5aadcc', 'compute_asf-a20d2ae6-b52e-4145-92d9-535accb3a837', 'compute_asf-6e2daa83-d4fc-4279-90e4-9b0d8f5e1c74', 'compute_asf-90c491ae-e65a-4669-9b77-b14080881892', 'compute_asf-fe0e74c6-458c-4741-8103-400a80883d1c', 'compute_asf-49cb8894-b264-43a5-95da-d08f19dc52c1', 'compute_asf-ca9ea9ba-8191-40a7-a819-253daa6dde23', 'compute_asf-f5d1da04-dc33-49a5-a282-635e97a0dd50', 'compute_asf-d351232f-5ea7-4757-9a3f-e29e033c7fe8', 'compute_asf-93776c4d-af26-4f96-a219-2797bab58296', 'compute_asf-87f5904a-705e-454a-b9f1-d3a0956f03a1', 'compute_asf-ac9297bb-2011-4c60-a5e6-a14b2981efa1', 'compute_asf-765ef81a-2685-4852-ab30-0f6575ab45a8', 'compute_asf-64d748c4-8d97-487f-a467-67c2b7887f8e', 'compute_asf-5549f846-5972-439

Computed ASFs
Formatted ASFs without scalar
Computed Bessel functions
Delayed coarse graining
Set up coarse graining
Computed coarse graining
Coarse graining formatted
Saved to data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-04-06/labradorsea/20260407_185732_SWOT_L3_labradorsea_cycle_036_timemean_removed_Bessels_tapered_CG.nc

Processing cycle: 037
Dask dashboard link: http://127.0.0.1:61695/status
Processing region: acc with reduce_xd_num: 1
Attempt 1/2 for cycle 037, region acc
Processing cycle 037 for region acc
Loading data from /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/forward/cycle_037/*.nc
Defined file paths with glob
Computing the maximum number of lines across all datasets
Number of datasets: 574
Maximum number of lines: 9860
Loading datasets and padding them to the maximum number of lines
Starting to compute padded datasets in parallel
Padded datasets
Filtered datasets by region
Removed time mean from dataset
Calculated advection
Computed ASFs
Formatted

...

Client restart failed (5/5 nanny worker(s) did not shut down within 120s: {'tcp://127.0.0.1:61733', 'tcp://127.0.0.1:61724', 'tcp://127.0.0.1:61730', 'tcp://127.0.0.1:61736', 'tcp://127.0.0.1:61721'}); creating a fresh client...
Created a fresh Dask client
Dask dashboard link: http://127.0.0.1:63582/status
Processing region: acc with reduce_xd_num: 1
Attempt 1/2 for cycle 042, region acc
Processing cycle 042 for region acc
Loading data from /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/forward/cycle_042/*.nc
Defined file paths with glob
Computing the maximum number of lines across all datasets
Number of datasets: 566
Maximum number of lines: 9860
Loading datasets and padding them to the maximum number of lines
Starting to compute padded datasets in parallel
Padded datasets
Filtered datasets by region
Removed time mean from dataset
Calculated advection
Computed ASFs
Formatted ASFs without scalar
Computed Bessel functions
Delayed coarse graining
Set up coars

2026-04-07 21:34:07,014 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,016 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,016 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,017 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,021 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,022 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,022 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:07,024 - distributed.nanny - WARNING - Worker process still alive after 4.0 seconds, killing
2026-04-07 21:34:08,015 - distributed.core - ERROR - Exception while handling op kill
Traceback (most recent call last):

Client restart failed (8/8 nanny worker(s) did not shut down within 120s: {'tcp://127.0.0.1:63624', 'tcp://127.0.0.1:63609', 'tcp://127.0.0.1:63603', 'tcp://127.0.0.1:63615', 'tcp://127.0.0.1:63621', 'tcp://127.0.0.1:63606', 'tcp://127.0.0.1:63612', 'tcp://127.0.0.1:63618'}); creating a fresh client...


2026-04-07 21:34:09,957 - distributed.deploy.spec - WARNING - Cluster closed without starting up


RuntimeError: Cluster failed to start: [Errno 28] No space left on device: '/var/folders/3y/g7f_qxmd57j05ln83z96smdr0000gn/T/dask-scratch-space/tmpstanvzch'

# 2026-05-07 SWOT L3 fast

In [ ]:
import gc
import os
import re
import glob
import time
import signal
import warnings
import logging
import xarray as xr

# Configure Dask memory settings
dask.config.set({
    "distributed.worker.memory.target": 0.7,
    "distributed.worker.memory.spill": 0.8,
    "distributed.worker.memory.pause": 0.85,
    "distributed.worker.memory.terminate": 0.95,
    "logging.distributed": "error",
})

def create_client():
    """Helper function to create a new Dask client."""
    return Client(
        n_workers=8,
        threads_per_worker=1,
        memory_limit="40GB",
        silence_logs=logging.WARNING,
    )

def reset_client(client):
    """Safely reset Dask client without crashing on teardown timeouts."""
    try:
        client.shutdown()
    except Exception as shutdown_err:
        print(f"Client shutdown warning (safe to ignore): {shutdown_err}")

    try:
        client.close()
    except Exception as close_err:
        print(f"Client close warning (safe to ignore): {close_err}")

    gc.collect()
    new_client = create_client()
    print("Created a fresh Dask client")
    return new_client

class RunTimeoutError(TimeoutError):
    pass

def _timeout_handler(signum, frame):
    raise RunTimeoutError("Run exceeded timeout window without completion")

# macOS supports SIGALRM (this notebook appears to run on macOS)
signal.signal(signal.SIGALRM, _timeout_handler)

# ---------- Tunables ----------
STALL_TIMEOUT_MIN = 20          # hard timeout per region/cycle attempt
STALL_TIMEOUT_SEC = STALL_TIMEOUT_MIN * 60
MAX_RETRIES = 2                 # retries per region/cycle before skip
run_new = False                 # False => skip existing completed outputs
# -----------------------------

USE_TRY_EXCEPT = True
client = create_client()

region_skips = [
    # "acc", "already processed",
    # "nwpacific", "already processed",
    # "capebasin", "weird error",
    # "labradorsea", "too large, redo with fewer cycles",
    # "nwaustralia", "already processed",
    # "westatlantic", "too large, redo with fewer cycles",
    # "newcaledonia", "weird error",
]

SWOT_PHASE = "fast_phase" # options: "fast_phase", "science_phase"

problematic_region_cycles = []
problematic_log_path = "problematic_region_cycles.log"

parent_dir = "/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0"
output_dir = f"data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-07/{SWOT_PHASE}"

mean_passstr = "474-578" if SWOT_PHASE == "fast_phase" else "001-048"
mean_filepath = f"/Volumes/Promise Disk/data/validated_swot/*Expert_Pass*_Mean_cycles-{mean_passstr}_v3.0.nc"

ASF = True
LLL = True
CG = True
scalar = None
Bessels = True
timemean_removal_method = "cycle_mean"  # options: None, "cycle_mean", "linear_fit"
cleaned = False
taper_SF = True
flip_swath = False
coarsen = 5 

cycles = [str(c).zfill(3) for c in range(474, 579)] if SWOT_PHASE == "fast_phase" else [str(c).zfill(3) for c in range(1, 49)]

for idx, cycnum in enumerate(cycles):

    try:
        glob_pattern = f"{parent_dir}/reproc/cycle_{cycnum}/*.nc"
        
        # periodic proactive reset
        if idx > 0 and idx % 10 == 0:
            print(f"Restarting Dask client after {idx} iterations...")
            client = reset_client(client)

        for region, params in region_dict.items():
            if region in region_skips:
                print(f"Skipping region: {region}, {region_skips[region_skips.index(region) + 1]}")
                continue

            reduce_xd_num = params["reduce_xd_num"]
            lat_north = params["lat_north"]
            lat_south = params["lat_south"]
            lon_east = params["lon_east"]
            lon_west = params["lon_west"]

            # convert negative longitudes to 0..360
            if lon_west < 0:
                lon_west += 360
            if lon_east < 0:
                lon_east += 360

            print(f"Processing region: {region}")

            region_output_dir = os.path.join(output_dir, region)
            os.makedirs(region_output_dir, exist_ok=True)

            suffix = ""
            suffix += f"_{region}_cycle_{cycnum}"
            suffix += f"_coarsen_to_1div{(48 / coarsen):.1f}deg"
            suffix += f"_timemean_removed_{timemean_removal_method}" if timemean_removal_method else ""
            suffix += "_cleaned" if cleaned else ""
            suffix += "_flipped_swath" if flip_swath else ""
            suffix += f"{scalar}" if scalar else ""
            suffix += "_LLL" if LLL else ""
            suffix += "_Bessels" if Bessels else ""
            suffix += "_tapered" if taper_SF else ""
            suffix += "_CG" if CG else ""
            print(f"Output filename suffix: {suffix}")

            def matching_outputs():
                return [
                    f for f in os.listdir(region_output_dir)
                    if (
                        f.endswith(".nc")
                        and suffix in f
                        and f"SWOT_L3_{region}_cycle_{cycnum}" in f
                    )
                ]

            # skip completed
            if not run_new:
                existing_files = matching_outputs()
                if existing_files:
                    print(f"Skipping cycle {cycnum} for region {region} as output file already exists: {existing_files[0]}")
                    continue
            print(f"Client dashboard link: {client.dashboard_link}")
            xsfuncs.compute_and_save_SWOT_L3(
                        filepath=glob_pattern,
                        mean_filepath=mean_filepath,
                        outpath=output_dir,
                        region_name=region,
                        cycle_num=cycnum,
                        lat_north=lat_north,
                        lat_south=lat_south,
                        lon_east=lon_east,
                        lon_west=lon_west,
                        # ASF=ASF,
                        LLL=LLL,
                        CG=CG,
                        reduce_xd_num=reduce_xd_num,
                        scalar=scalar,
                        Bessels=Bessels,
                        timemean_removal_method=timemean_removal_method,
                        taper_SF=taper_SF,
                        coarsen=coarsen,
                    )
    except xr.AlignmentError as e:
        print(f"AlignmentError for region {region}, cycle {cycnum}: {e}")
        print("This may be due to mismatched dimensions in the input files. Skipping this region and cycle.")
        reason = f"alignmenterror: {e}"
        problematic_region_cycles.append((cycnum, region, reason))
        with open(problematic_log_path, "a") as log_file:
            log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")
    except TypeError as e:
        print(f"TypeError for region {region}, cycle {cycnum}: {e}")
 
        reason = f"typeerror: {e}"
        print(f"Skipping cycle {cycnum} for region {region}: {reason}")
        problematic_region_cycles.append((cycnum, region, reason))
        with open(problematic_log_path, "a") as log_file:
            log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")

    except IndexError as e:
        print(f"IndexError for region {region}, cycle {cycnum}: {e}")
 
        reason = f"indexerror: {e}"
        print(f"Skipping cycle {cycnum} for region {region}: {reason}")
        problematic_region_cycles.append((cycnum, region, reason))
        with open(problematic_log_path, "a") as log_file:
            log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")

Processing region: acc
Output filename suffix: _acc_cycle_474_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG
Skipping cycle 474 for region acc as output file already exists: 20260507_165949_SWOT_L3_acc_cycle_474_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG.nc
Processing region: nwpacific
Output filename suffix: _nwpacific_cycle_474_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG
Skipping cycle 474 for region nwpacific as output file already exists: 20260507_165953_SWOT_L3_nwpacific_cycle_474_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG.nc
Processing region: capebasin
Output filename suffix: _capebasin_cycle_474_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG
Client dashboard link: http://127.0.0.1:59255/status
Loading data from /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_474/*.nc
Defined file paths with glob
Num

# 2026-05-05 SWOT L3 science

In [ ]:
import gc
import os
import re
import glob
import time
import signal
import warnings
import logging
import xarray as xr

# Configure Dask memory settings
dask.config.set({
    "distributed.worker.memory.target": 0.7,
    "distributed.worker.memory.spill": 0.8,
    "distributed.worker.memory.pause": 0.85,
    "distributed.worker.memory.terminate": 0.95,
    "logging.distributed": "error",
})

def create_client():
    """Helper function to create a new Dask client."""
    return Client(
        n_workers=8,
        threads_per_worker=1,
        memory_limit="40GB",
        silence_logs=logging.WARNING,
    )

def reset_client(client):
    """Safely reset Dask client without crashing on teardown timeouts."""
    try:
        client.shutdown()
    except Exception as shutdown_err:
        print(f"Client shutdown warning (safe to ignore): {shutdown_err}")

    try:
        client.close()
    except Exception as close_err:
        print(f"Client close warning (safe to ignore): {close_err}")

    gc.collect()
    new_client = create_client()
    print("Created a fresh Dask client")
    return new_client

class RunTimeoutError(TimeoutError):
    pass

def _timeout_handler(signum, frame):
    raise RunTimeoutError("Run exceeded timeout window without completion")

# macOS supports SIGALRM (this notebook appears to run on macOS)
signal.signal(signal.SIGALRM, _timeout_handler)

# ---------- Tunables ----------
STALL_TIMEOUT_MIN = 20          # hard timeout per region/cycle attempt
STALL_TIMEOUT_SEC = STALL_TIMEOUT_MIN * 60
MAX_RETRIES = 2                 # retries per region/cycle before skip
run_new = False                 # False => skip existing completed outputs
# -----------------------------

USE_TRY_EXCEPT = True
client = create_client()

region_skips = [
    # "acc", "already processed",
    # "nwpacific", "already processed",
    # "capebasin", "already processed",
    "labradorsea", "too large, redo with fewer cycles",
    # "nwaustralia", "already processed",
    "westatlantic", "too large, redo with fewer cycles",
    # "newcaledonia", "already processed",
]

SWOT_PHASE = "science_phase" # options: "fast_phase", "science_phase"

problematic_region_cycles = []
problematic_log_path = "problematic_region_cycles.log"

parent_dir = "/Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0"
output_dir = f"data/SWOT_L3/SWOT_L3_LR_SSH_3.0/2026-05-07/{SWOT_PHASE}"

mean_filepath = "/Volumes/Promise Disk/data/validated_swot/*Expert_Pass*_Mean_cycles-001-048_v3.0.nc"

ASF = True
LLL = True
CG = True
scalar = None
Bessels = True
timemean_removal_method = "cycle_mean"  # options: None, "cycle_mean", "linear_fit"
cleaned = False
taper_SF = True
flip_swath = False
coarsen = 5 

cycles = [str(c).zfill(3) for c in range(1, 49)]

for idx, cycnum in enumerate(cycles):

    try:
        if int(cycnum) > 1 or int(cycnum) < 32:

            glob_pattern = f"{parent_dir}/reproc/cycle_{cycnum}/*.nc"
        
        if int(cycnum) >= 33 and int(cycnum) <= 48:

            glob_pattern = f"{parent_dir}/forward/cycle_{cycnum}/*.nc"

        if int(cycnum) == 32:

            glob_pattern = f"{parent_dir}/*/cycle_{cycnum}/*.nc"

        # periodic proactive reset
        if idx > 0 and idx % 10 == 0:
            print(f"Restarting Dask client after {idx} iterations...")
            client = reset_client(client)

        for region, params in region_dict.items():
            if region in region_skips:
                print(f"Skipping region: {region}, {region_skips[region_skips.index(region) + 1]}")
                continue

            reduce_xd_num = params["reduce_xd_num"]
            lat_north = params["lat_north"]
            lat_south = params["lat_south"]
            lon_east = params["lon_east"]
            lon_west = params["lon_west"]

            # convert negative longitudes to 0..360
            if lon_west < 0:
                lon_west += 360
            if lon_east < 0:
                lon_east += 360

            print(f"Processing region: {region}")

            region_output_dir = os.path.join(output_dir, region)
            os.makedirs(region_output_dir, exist_ok=True)

            suffix = ""
            suffix += f"_{region}_cycle_{cycnum}"
            suffix += f"_coarsen_to_1div{(48 / coarsen):.1f}deg"
            suffix += f"_timemean_removed_{timemean_removal_method}" if timemean_removal_method else ""
            suffix += "_cleaned" if cleaned else ""
            suffix += "_flipped_swath" if flip_swath else ""
            suffix += f"{scalar}" if scalar else ""
            suffix += "_LLL" if LLL else ""
            suffix += "_Bessels" if Bessels else ""
            suffix += "_tapered" if taper_SF else ""
            suffix += "_CG" if CG else ""
            print(f"Output filename suffix: {suffix}")

            def matching_outputs():
                return [
                    f for f in os.listdir(region_output_dir)
                    if (
                        f.endswith(".nc")
                        and suffix in f
                        and f"SWOT_L3_{region}_cycle_{cycnum}" in f
                    )
                ]

            # skip completed
            if not run_new:
                existing_files = matching_outputs()
                if existing_files:
                    print(f"Skipping cycle {cycnum} for region {region} as output file already exists: {existing_files[0]}")
                    continue
            print(f"Client dashboard link: {client.dashboard_link}")
            xsfuncs.compute_and_save_SWOT_L3(
                        filepath=glob_pattern,
                        mean_filepath=mean_filepath,
                        outpath=output_dir,
                        region_name=region,
                        cycle_num=cycnum,
                        lat_north=lat_north,
                        lat_south=lat_south,
                        lon_east=lon_east,
                        lon_west=lon_west,
                        # ASF=ASF,
                        LLL=LLL,
                        CG=CG,
                        reduce_xd_num=reduce_xd_num,
                        scalar=scalar,
                        Bessels=Bessels,
                        timemean_removal_method=timemean_removal_method,
                        taper_SF=taper_SF,
                        coarsen=coarsen,
                    )
    except xr.AlignmentError as e:
        print(f"AlignmentError for region {region}, cycle {cycnum}: {e}")
        print("This may be due to mismatched dimensions in the input files. Skipping this region and cycle.")
        reason = f"alignmenterror: {e}"
        problematic_region_cycles.append((cycnum, region, reason))
        with open(problematic_log_path, "a") as log_file:
            log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")
    except TypeError as e:
        print(f"TypeError for region {region}, cycle {cycnum}: {e}")
 
        reason = f"typeerror: {e}"
        print(f"Skipping cycle {cycnum} for region {region}: {reason}")
        problematic_region_cycles.append((cycnum, region, reason))
        with open(problematic_log_path, "a") as log_file:
            log_file.write(f"cycle={cycnum}, region={region}, reason={reason}\n")

Processing region: acc
Output filename suffix: _acc_cycle_001_coarsen_to_1div9.6deg_timemean_removed_cycle_mean_LLL_Bessels_tapered_CG
Client dashboard link: http://127.0.0.1:57152/status
Loading data from /Volumes/Promise Disk/data/validated_swot/SWOT_L3_LR_SSH/SWOT_L3_LR_SSH_3.0/reproc/cycle_001/*.nc
Defined file paths with glob
Number of datasets: 409
Loading datasets and padding them to the maximum number of lines


# Testing